# HAHA 2026 — Retrieval + Stacking pipeline (Kaggle)

Pipeline cho 2 subtask của **HAHA at IberLEF 2026**. Đã chỉnh cho dữ liệu train đã gộp (`trial + dev_gold`) đặt trong Kaggle Datasets.

- **Subtask 1** — `satirical` / `real`  → F1 lớp `satirical`  · chia **StratifiedKFold(10)**
- **Subtask 2** — `machine` / `human`  → F1 lớp `machine`     · chia **GroupKFold theo `headline`**

> Bật GPU: *Settings → Accelerator → GPU*. Bật Internet: *Settings → Internet → On* (để pip install và tải model HuggingFace).

## 1. Cài đặt
Kaggle đã có sẵn `torch`, `transformers`, `accelerate`, `lightgbm`. Chỉ cần thêm `sentence-transformers` (cho nhánh LLM, tùy chọn).

In [12]:
!pip -q install -U "transformers>=4.38,<5" "accelerate>=0.26" sentencepiece sentence-transformers

## 2. Chọn subtask & định vị dữ liệu
Đổi `SUBTASK` = 1 hoặc 2. Ô này tự tìm file `task*_train.tsv` / `task*_test.tsv` trong `/kaggle/input` (không cần sửa tên dataset).
File train của bạn **đã gộp trial + dev_gold** nên không cộng thêm dev nữa. Nếu chưa upload file test, notebook sẽ chỉ chạy CV và báo F1 (bỏ qua bước nộp).

In [13]:
import glob, os

def find(name):
    hits = sorted(glob.glob(f"/kaggle/input/**/{name}", recursive=True))
    return hits[0] if hits else name  # fallback: ten file truc tiep

SUBTASK = 2# <-- doi thanh 2 cho subtask 2

if SUBTASK == 1:
    TRAIN_PATHS = [find("task1_train.tsv")]
    TEST_PATH   = find("task1_test.tsv")
    TEXT_COLS, LABEL_COL = ["headline", "context"], "tag"
    POS, OUT_POS, OUT_NEG = "satirical", "satirical", "real"
    GROUP_COL, N_FOLDS, MAX_LEN = None, 10, 128
    USE_DETECTOR = False
    USE_TFIDF, EPOCHS = True, 5     # task1: TF-IDF manh hon encoder -> ensemble
else:
    TRAIN_PATHS = [find("task2_train.tsv")]
    TEST_PATH   = find("task2_test.tsv")
    TEXT_COLS, LABEL_COL = ["headline", "joke"], "tag"
    POS, OUT_POS, OUT_NEG = "machine", "machine", "human"
    GROUP_COL, N_FOLDS, MAX_LEN = "headline", 5, 192
    USE_DETECTOR = True
    USE_TFIDF, EPOCHS = False, 4    # task2: encoder+detector da manh, TF-IDF lam loang

print("subtask", SUBTASK)
print("train:", TRAIN_PATHS)
print("test :", TEST_PATH if os.path.exists(TEST_PATH) else "(chua co -> chi chay CV, bo qua nop)")
print("tfidf:", USE_TFIDF, "| detector:", USE_DETECTOR, "| epochs:", EPOCHS)

subtask 2
train: ['/kaggle/input/datasets/trongnhantran25/dataaaa/train_data/task2_train.tsv']
test : /kaggle/input/datasets/trongnhantran25/dataaaa/train_data/task2_test.tsv


## 3. Kiểm tra nhanh dữ liệu

In [14]:
import csv, pandas as pd
_df = pd.read_csv(TRAIN_PATHS[0], sep="\t", dtype=str, keep_default_na=False, quoting=csv.QUOTE_MINIMAL)
print("rows:", len(_df), "| cols:", _df.columns.tolist())
print("label dist:", _df[LABEL_COL].value_counts().to_dict())
if GROUP_COL:
    print(f"unique {GROUP_COL}:", _df[GROUP_COL].nunique(), "(GroupKFold se chia theo cot nay)")
print("vi du:", {k: (v[:60] if isinstance(v,str) else v) for k,v in _df.iloc[0].to_dict().items()})

rows: 362 | cols: ['id', 'headline', 'joke', 'tag']
label dist: {'machine': 181, 'human': 181}
unique headline: 41 (GroupKFold se chia theo cot nay)
vi du: {'id': 't2_0001', 'headline': 'Restaurante vegano de Nueva York volverá a servir carne para', 'joke': 'Un restaurante vegano en Nueva York va a empezar a servir ca', 'tag': 'machine'}


## 4. Cấu hình

In [15]:
from dataclasses import dataclass, field
from typing import List, Optional

@dataclass
class Config:
    subtask: int = 1
    train_paths: List[str] = field(default_factory=lambda: TRAIN_PATHS)
    dev_path: Optional[str] = None              # train da gop trial+dev_gold roi
    test_path: str = ""

    id_col: str = "id"
    text_cols: List[str] = field(default_factory=lambda: ["headline"])
    label_col: str = "tag"
    group_col: Optional[str] = None             # task2 -> "headline"

    positive_label: str = "satirical"
    out_positive: str = "satirical"
    out_negative: str = "real"
    out_label_col: str = "tag"

    use_encoder_branch: bool = True
    use_detection_branch: bool = False
    use_tfidf_branch: bool = False
    use_llm_branch: bool = False

    encoder_model: str = "PlanTL-GOB-ES/roberta-large-bne"
    ppl_model: str = "DeepESP/gpt2-spanish"
    retriever_model: str = "intfloat/multilingual-e5-large"
    llm_model: str = "google/gemma-2-9b-it"

    n_folds: int = 5
    epochs: int = 4
    encoder_seeds: List[int] = field(default_factory=lambda: [42])  # them seed -> trung binh
    lr: float = 2e-5
    batch_size: int = 16
    max_len: int = 128
    seed: int = 42

    few_shot_k_per_class: int = 3
    self_consistency_n: int = 5

    output_dir: str = "/kaggle/working/outputs"
    submission_csv: str = "submission.csv"
    make_zip: bool = True
    save_checkpoints: bool = True              # xuat checkpoint trong so tot nhat
    checkpoint_dir: Optional[str] = None       # mac dinh: <output_dir>/checkpoints

CFG = Config(
    subtask=SUBTASK, train_paths=TRAIN_PATHS, dev_path=None, test_path=TEST_PATH,
    id_col="id", text_cols=TEXT_COLS, label_col=LABEL_COL, group_col=GROUP_COL,
    positive_label=POS, out_positive=OUT_POS, out_negative=OUT_NEG, out_label_col="tag",
    n_folds=N_FOLDS, max_len=MAX_LEN, epochs=EPOCHS,
    use_encoder_branch=True, use_detection_branch=USE_DETECTOR,
    use_tfidf_branch=USE_TFIDF, use_llm_branch=False,
)
print("Subtask", CFG.subtask, "| folds:", CFG.n_folds,
      "| group:", CFG.group_col, "| encoder:", CFG.encoder_model)

Subtask 2 | folds: 5 | group: headline | encoder: PlanTL-GOB-ES/roberta-large-bne


In [16]:
# ==== FIX encoder: dung model co fast tokenizer on dinh ====
# xlm-roberta manh cho tieng TBN va co tokenizer.json -> tranh loi slow-tokenizer
CFG.encoder_model = "xlm-roberta-base"      # nhe, chac chan chay
# CFG.encoder_model = "xlm-roberta-large"   # manh hon; nho giam batch + tang max_len:
# CFG.batch_size = 8
# CFG.encoder_model = "dccuchile/bert-base-spanish-wwm-cased"  # BETO (cung rat on dinh)
# CFG.encoder_seeds = [42, 1, 2]   # chay nhieu seed roi trung binh: on dinh hon nhung lau hon

# Preflight: kiem tra tai duoc model (Kaggle phai bat Internet = On)
from transformers import AutoTokenizer
try:
    _ = AutoTokenizer.from_pretrained(CFG.encoder_model)
    print("Tokenizer OK:", CFG.encoder_model)
except Exception as e:
    raise RuntimeError(
        "Khong tai duoc model. Kiem tra: Settings -> Internet -> On "
        "(can tai khoan Kaggle da verify so dien thoai)."
    ) from e

Tokenizer OK: xlm-roberta-base


## 5. Imports & tiện ích
Điểm mới: `make_folds` dùng **StratifiedGroupKFold** khi có `group_col` (task2), giữ mỗi headline gọn trong một fold.

## 5.0. Thư viện Machine Learning **tự cài đặt** (thay cho scikit-learn)

Toàn bộ phần ML cổ điển được code tay bằng NumPy (chỉ mượn `scipy.sparse` làm cấu trúc ma trận thưa cho TF-IDF). Bao gồm: `f1_score`, `StratifiedKFold`, `StratifiedGroupKFold`, `LogisticRegression` (Adam + L2 + class_weight), `TfidfVectorizer` (word & char_wb n-grams) và `GradientBoostingClassifier` (Newton boosting kiểu XGBoost).


In [ ]:
# ============================================================================
# Thư viện Machine Learning tự cài đặt (không dùng scikit-learn)
# Chỉ phụ thuộc numpy (+ scipy.sparse cho ma trận thưa TF-IDF).
# ============================================================================
import numpy as np
import unicodedata, re
from collections import defaultdict

try:
    from scipy.sparse import csr_matrix, hstack as _sp_hstack, issparse
    _HAS_SCIPY = True
except Exception:
    _HAS_SCIPY = False


# ----------------------------------------------------------------------------
# 1) Metric: F1
# ----------------------------------------------------------------------------
def f1_score(y_true, y_pred, pos_label=1, zero_division=0):
    y_true = np.asarray(y_true); y_pred = np.asarray(y_pred)
    tp = int(np.sum((y_pred == pos_label) & (y_true == pos_label)))
    fp = int(np.sum((y_pred == pos_label) & (y_true != pos_label)))
    fn = int(np.sum((y_pred != pos_label) & (y_true == pos_label)))
    denom = 2 * tp + fp + fn
    if denom == 0:
        return float(zero_division)
    return 2.0 * tp / denom


# ----------------------------------------------------------------------------
# 2) Cross-validation splitters
# ----------------------------------------------------------------------------
class StratifiedKFold:
    """Chia fold giữ nguyên tỉ lệ lớp (giống sklearn ở mức phân phối)."""
    def __init__(self, n_splits=5, shuffle=False, random_state=None):
        self.n_splits = n_splits; self.shuffle = shuffle; self.random_state = random_state

    def split(self, X, y, groups=None):
        y = np.asarray(y)
        n = len(y)
        rng = np.random.RandomState(self.random_state)
        test_fold = np.empty(n, dtype=int)
        for cls in np.unique(y):
            idx = np.where(y == cls)[0]
            if self.shuffle:
                rng.shuffle(idx)
            # rải vòng tròn -> mỗi fold nhận ~ bằng nhau số mẫu của lớp này
            test_fold[idx] = np.arange(len(idx)) % self.n_splits
        for f in range(self.n_splits):
            va = np.where(test_fold == f)[0]
            tr = np.where(test_fold != f)[0]
            yield tr, va


class StratifiedGroupKFold:
    """Vừa stratify theo nhãn, vừa giữ nguyên cả group trong cùng một fold.
    Cài đặt tham lam giống thuật toán của sklearn."""
    def __init__(self, n_splits=5, shuffle=False, random_state=None):
        self.n_splits = n_splits; self.shuffle = shuffle; self.random_state = random_state

    def split(self, X, y, groups):
        y = np.asarray(y); groups = np.asarray(groups)
        _, y_enc = np.unique(y, return_inverse=True)
        n_classes = int(y_enc.max()) + 1
        uniq_g, g_inv = np.unique(groups, return_inverse=True)
        n_groups = len(uniq_g)

        group_y = np.zeros((n_groups, n_classes), dtype=float)
        for gi, yi in zip(g_inv, y_enc):
            group_y[gi, yi] += 1
        y_cnt = group_y.sum(axis=0)
        y_cnt[y_cnt == 0] = 1.0  # tránh chia 0

        rng = np.random.RandomState(self.random_state)
        order = np.arange(n_groups)
        if self.shuffle:
            rng.shuffle(order)
        # ưu tiên group "lệch" nhất trước (độ lệch chuẩn lớn)
        order = sorted(order, key=lambda g: -np.std(group_y[g]))

        fold_y = np.zeros((self.n_splits, n_classes), dtype=float)
        g2f = np.full(n_groups, -1, dtype=int)
        for g in order:
            best_f, best_eval = -1, None
            for f in range(self.n_splits):
                fold_y[f] += group_y[g]
                std = np.std(fold_y / y_cnt.reshape(1, -1), axis=0).mean()
                fold_y[f] -= group_y[g]
                if best_eval is None or std < best_eval:
                    best_eval, best_f = std, f
            fold_y[best_f] += group_y[g]
            g2f[g] = best_f

        fold_of_sample = g2f[g_inv]
        for f in range(self.n_splits):
            va = np.where(fold_of_sample == f)[0]
            tr = np.where(fold_of_sample != f)[0]
            yield tr, va


# ----------------------------------------------------------------------------
# 3) Logistic Regression (L2, class_weight='balanced'), tối ưu bằng Adam.
#    Hỗ trợ cả ma trận numpy dày lẫn scipy.sparse (cho TF-IDF).
# ----------------------------------------------------------------------------
def _sigmoid(z):
    out = np.empty_like(z, dtype=float)
    pos = z >= 0
    out[pos] = 1.0 / (1.0 + np.exp(-z[pos]))
    ez = np.exp(z[~pos])
    out[~pos] = ez / (1.0 + ez)
    return out


class LogisticRegression:
    def __init__(self, C=1.0, max_iter=1000, class_weight=None,
                 fit_intercept=True, lr=0.5, tol=1e-6, random_state=0):
        self.C = C; self.max_iter = max_iter; self.class_weight = class_weight
        self.fit_intercept = fit_intercept; self.lr = lr; self.tol = tol
        self.random_state = random_state

    def fit(self, X, y):
        y = np.asarray(y).astype(float)
        n, d = X.shape
        sparse = _HAS_SCIPY and issparse(X)
        if sparse:
            X = X.tocsr(); Xt = X.T.tocsr()

        # sample weights
        if self.class_weight == "balanced":
            classes = np.unique(y); ncl = len(classes)
            cw = {c: n / (ncl * np.sum(y == c)) for c in classes}
            sw = np.array([cw[v] for v in y], dtype=float)
        elif isinstance(self.class_weight, dict):
            sw = np.array([self.class_weight.get(v, 1.0) for v in y], dtype=float)
        else:
            sw = np.ones(n, dtype=float)

        w = np.zeros(d, dtype=float); b = 0.0
        lam = 1.0 / self.C
        # Adam state
        mw = np.zeros(d); vw = np.zeros(d); mb = 0.0; vb = 0.0
        b1, b2, eps = 0.9, 0.999, 1e-8
        prev = np.inf
        for t in range(1, self.max_iter + 1):
            z = (X @ w) + b
            p = _sigmoid(z)
            r = sw * (p - y)                       # residual có trọng số
            gw = (Xt @ r if sparse else X.T @ r) + lam * w
            gb = r.sum() if self.fit_intercept else 0.0

            mw = b1 * mw + (1 - b1) * gw; vw = b2 * vw + (1 - b2) * (gw * gw)
            mhat = mw / (1 - b1 ** t);    vhat = vw / (1 - b2 ** t)
            w -= self.lr * mhat / (np.sqrt(vhat) + eps)
            if self.fit_intercept:
                mb = b1 * mb + (1 - b1) * gb; vb = b2 * vb + (1 - b2) * (gb * gb)
                mbh = mb / (1 - b1 ** t);     vbh = vb / (1 - b2 ** t)
                b -= self.lr * mbh / (np.sqrt(vbh) + eps)

            if t % 25 == 0:
                eps2 = 1e-12
                loss = np.sum(sw * (-y * np.log(p + eps2) - (1 - y) * np.log(1 - p + eps2))) \
                       + 0.5 * lam * np.dot(w, w)
                if abs(prev - loss) < self.tol * max(1.0, abs(prev)):
                    break
                prev = loss

        self.coef_ = w.reshape(1, -1)
        self.intercept_ = np.array([b])
        self.classes_ = np.array([0, 1])
        return self

    def decision_function(self, X):
        return (X @ self.coef_[0]) + self.intercept_[0]

    def predict_proba(self, X):
        p = _sigmoid(self.decision_function(X))
        return np.column_stack([1 - p, p])

    def predict(self, X):
        return (self.predict_proba(X)[:, 1] >= 0.5).astype(int)


# ----------------------------------------------------------------------------
# 4) TF-IDF Vectorizer (word + char_wb n-grams), trả về scipy.sparse CSR.
#    Tương thích sklearn: smooth_idf, sublinear_tf, strip_accents, l2 norm.
# ----------------------------------------------------------------------------
def _strip_accents_unicode(s):
    nf = unicodedata.normalize("NFKD", s)
    return "".join(c for c in nf if not unicodedata.combining(c))


class TfidfVectorizer:
    _token_re = re.compile(r"(?u)\b\w\w+\b")

    def __init__(self, analyzer="word", ngram_range=(1, 1), min_df=1,
                 sublinear_tf=False, strip_accents=None, lowercase=True, norm="l2"):
        self.analyzer = analyzer; self.ngram_range = ngram_range
        self.min_df = min_df; self.sublinear_tf = sublinear_tf
        self.strip_accents = strip_accents; self.lowercase = lowercase; self.norm = norm

    def _preprocess(self, doc):
        doc = str(doc)
        if self.strip_accents == "unicode":
            doc = _strip_accents_unicode(doc)
        if self.lowercase:
            doc = doc.lower()
        return doc

    def _word_ngrams(self, doc):
        toks = self._token_re.findall(doc)
        lo, hi = self.ngram_range
        out = []
        if lo <= 1:
            out.extend(toks)
        ntok = len(toks)
        for n in range(max(lo, 2), hi + 1):
            for i in range(ntok - n + 1):
                out.append(" ".join(toks[i:i + n]))
        return out

    def _char_wb_ngrams(self, doc):
        lo, hi = self.ngram_range
        out = []
        for tok in doc.split():
            w = " " + tok + " "
            L = len(w)
            for n in range(lo, hi + 1):
                out.append(w[0:n])
                offset = 0
                while offset + n < L:
                    offset += 1
                    out.append(w[offset:offset + n])
                if offset == 0:   # từ vừa/ngắn hơn n -> chỉ tính 1 lần rồi dừng (giống sklearn)
                    break
        return out

    def _analyze(self, raw):
        doc = self._preprocess(raw)
        if self.analyzer == "word":
            return self._word_ngrams(doc)
        elif self.analyzer == "char_wb":
            return self._char_wb_ngrams(doc)
        raise ValueError("analyzer phải là 'word' hoặc 'char_wb'")

    def fit_transform(self, raw_docs):
        raw_docs = list(raw_docs)
        analyzed = [self._analyze(d) for d in raw_docs]
        df = defaultdict(int)
        for terms in analyzed:
            for t in set(terms):
                df[t] += 1
        vocab = sorted(t for t, c in df.items() if c >= self.min_df)
        self.vocabulary_ = {t: i for i, t in enumerate(vocab)}
        n_docs = len(raw_docs)
        dfv = np.array([df[t] for t in vocab], dtype=float)
        # smooth_idf = True (mặc định sklearn)
        self.idf_ = np.log((1.0 + n_docs) / (1.0 + dfv)) + 1.0
        return self._to_matrix(analyzed)

    def transform(self, raw_docs):
        analyzed = [self._analyze(d) for d in raw_docs]
        return self._to_matrix(analyzed)

    def _to_matrix(self, analyzed):
        V = len(self.vocabulary_)
        indptr = [0]; indices = []; data = []
        for terms in analyzed:
            counts = defaultdict(int)
            for t in terms:
                j = self.vocabulary_.get(t)
                if j is not None:
                    counts[j] += 1
            if counts:
                cols = np.fromiter(counts.keys(), dtype=int)
                tf = np.fromiter(counts.values(), dtype=float)
                if self.sublinear_tf:
                    tf = 1.0 + np.log(tf)
                vals = tf * self.idf_[cols]
                if self.norm == "l2":
                    nrm = np.sqrt(np.dot(vals, vals))
                    if nrm > 0:
                        vals = vals / nrm
                indices.extend(cols.tolist()); data.extend(vals.tolist())
            indptr.append(len(indices))
        if not _HAS_SCIPY:
            raise RuntimeError("Cần scipy.sparse cho TF-IDF.")
        return csr_matrix((np.array(data), np.array(indices), np.array(indptr)),
                          shape=(len(analyzed), V))


# ----------------------------------------------------------------------------
# 5) Gradient Boosting (cây hồi quy) cho phân loại nhị phân — kiểu Newton/XGBoost.
#    Thay cho LightGBM / HistGradientBoostingClassifier.
# ----------------------------------------------------------------------------
class _Node:
    __slots__ = ("feat", "thr", "left", "right", "value")
    def __init__(self):
        self.feat = -1; self.thr = 0.0; self.left = None; self.right = None; self.value = 0.0


class _RegTree:
    """Cây hồi quy tối ưu logloss qua gradient/hessian (gain kiểu XGBoost)."""
    def __init__(self, max_depth=3, min_samples_leaf=20, reg_lambda=1.0,
                 min_gain=1e-6, colsample=1.0, rng=None):
        self.max_depth = max_depth; self.min_samples_leaf = min_samples_leaf
        self.reg_lambda = reg_lambda; self.min_gain = min_gain
        self.colsample = colsample; self.rng = rng or np.random.RandomState(0)

    def fit(self, X, grad, hess):
        self.n_features_ = X.shape[1]
        self.root = self._build(X, grad, hess, depth=0)
        return self

    def _leaf_value(self, g, h):
        return -g.sum() / (h.sum() + self.reg_lambda)

    def _build(self, X, g, h, depth):
        node = _Node()
        node.value = self._leaf_value(g, h)
        n = X.shape[0]
        if depth >= self.max_depth or n < 2 * self.min_samples_leaf:
            return node

        G, H = g.sum(), h.sum()
        parent_score = (G * G) / (H + self.reg_lambda)
        best_gain, best_feat, best_thr, best_mask = self.min_gain, -1, 0.0, None

        n_feat = self.n_features_
        if self.colsample < 1.0:
            k = max(1, int(round(self.colsample * n_feat)))
            feats = self.rng.choice(n_feat, size=k, replace=False)
        else:
            feats = range(n_feat)

        for f in feats:
            xf = X[:, f]
            order = np.argsort(xf, kind="mergesort")
            xs = xf[order]; gs = g[order]; hs = h[order]
            Gl = np.cumsum(gs)[:-1]; Hl = np.cumsum(hs)[:-1]
            Gr = G - Gl; Hr = H - Hl
            # chỉ tách ở chỗ giá trị đổi (tránh tách giữa các điểm trùng)
            valid = xs[:-1] != xs[1:]
            # đủ số mẫu mỗi lá
            left_n = np.arange(1, n)
            valid &= (left_n >= self.min_samples_leaf) & ((n - left_n) >= self.min_samples_leaf)
            if not valid.any():
                continue
            gain = (Gl * Gl) / (Hl + self.reg_lambda) + (Gr * Gr) / (Hr + self.reg_lambda) - parent_score
            gain = np.where(valid, gain, -np.inf)
            k = int(np.argmax(gain))
            if gain[k] > best_gain:
                best_gain = gain[k]; best_feat = f
                best_thr = (xs[k] + xs[k + 1]) / 2.0
                best_mask = xf <= best_thr

        if best_feat < 0:
            return node
        node.feat = best_feat; node.thr = best_thr
        lm = best_mask; rm = ~best_mask
        node.left = self._build(X[lm], g[lm], h[lm], depth + 1)
        node.right = self._build(X[rm], g[rm], h[rm], depth + 1)
        return node

    def predict(self, X):
        out = np.empty(X.shape[0], dtype=float)
        for i in range(X.shape[0]):
            nd = self.root
            row = X[i]
            while nd.left is not None:
                nd = nd.left if row[nd.feat] <= nd.thr else nd.right
            out[i] = nd.value
        return out


class GradientBoostingClassifier:
    """Newton gradient boosting cho logloss nhị phân (tự cài, không sklearn)."""
    def __init__(self, n_estimators=400, learning_rate=0.05, max_depth=3,
                 min_samples_leaf=20, reg_lambda=1.0, subsample=1.0,
                 colsample_bytree=1.0, random_state=0, verbose=False):
        self.n_estimators = n_estimators; self.learning_rate = learning_rate
        self.max_depth = max_depth; self.min_samples_leaf = min_samples_leaf
        self.reg_lambda = reg_lambda; self.subsample = subsample
        self.colsample_bytree = colsample_bytree
        self.random_state = random_state; self.verbose = verbose

    def fit(self, X, y):
        X = np.ascontiguousarray(np.asarray(X, dtype=float))
        y = np.asarray(y, dtype=float)
        n = len(y)
        rng = np.random.RandomState(self.random_state)
        p0 = np.clip(y.mean(), 1e-6, 1 - 1e-6)
        self.base_ = np.log(p0 / (1 - p0))
        F = np.full(n, self.base_, dtype=float)
        self.trees_ = []
        for m in range(self.n_estimators):
            p = _sigmoid(F)
            grad = p - y
            hess = np.clip(p * (1 - p), 1e-6, None)
            if self.subsample < 1.0:
                k = max(self.min_samples_leaf * 2, int(self.subsample * n))
                idx = rng.choice(n, size=k, replace=False)
            else:
                idx = np.arange(n)
            tree = _RegTree(max_depth=self.max_depth, min_samples_leaf=self.min_samples_leaf,
                            reg_lambda=self.reg_lambda, colsample=self.colsample_bytree, rng=rng)
            tree.fit(X[idx], grad[idx], hess[idx])
            F += self.learning_rate * tree.predict(X)
            self.trees_.append(tree)
        return self

    def decision_function(self, X):
        X = np.ascontiguousarray(np.asarray(X, dtype=float))
        F = np.full(X.shape[0], self.base_, dtype=float)
        for tree in self.trees_:
            F += self.learning_rate * tree.predict(X)
        return F

    def predict_proba(self, X):
        p = _sigmoid(self.decision_function(X))
        return np.column_stack([1 - p, p])

    def predict(self, X):
        return (self.predict_proba(X)[:, 1] >= 0.5).astype(int)


In [17]:
import re, json, math, zipfile, random
import numpy as np
from typing import Dict, List, Tuple
# (KHONG dung scikit-learn) -> cac lop ML tu cai dat o muc 5.0 phia tren:
#   f1_score, StratifiedKFold, StratifiedGroupKFold, LogisticRegression,
#   TfidfVectorizer, GradientBoostingClassifier


def set_seed(seed):
    random.seed(seed); np.random.seed(seed); os.environ["PYTHONHASHSEED"] = str(seed)
    try:
        import torch
        torch.manual_seed(seed)
        if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
    except Exception:
        pass


def load_table(path):
    if not path or not os.path.exists(path):
        raise FileNotFoundError(f"Khong tim thay '{path}'.")
    return pd.read_csv(path, sep="\t", dtype=str, keep_default_na=False,
                       quoting=csv.QUOTE_MINIMAL)


def build_text(df, cfg):
    missing = [c for c in cfg.text_cols if c not in df.columns]
    if missing:
        raise KeyError(f"Thieu cot {missing}. Cot co san: {list(df.columns)}")
    cols = [df[c].astype(str).fillna("") for c in cfg.text_cols]
    if len(cols) == 1:
        return cols[0].tolist()
    return [" [SEP] ".join(p) for p in zip(*[c.tolist() for c in cols])]


def encode_labels(df, cfg):
    if cfg.label_col not in df.columns:
        raise KeyError(f"Thieu cot nhan '{cfg.label_col}'. Cot co san: {list(df.columns)}")
    raw = df[cfg.label_col].astype(str).str.strip().str.lower()
    y = (raw == cfg.positive_label.strip().lower()).astype(int).to_numpy()
    if y.sum() == 0:
        raise ValueError(f"Khong co dong khop positive_label='{cfg.positive_label}'. "
                         f"Nhan thay: {sorted(raw.unique())}")
    return y


def tune_threshold(y_true, prob_pos):
    best_t, best_f1 = 0.5, -1.0
    for t in np.linspace(0.05, 0.95, 181):
        f1 = f1_score(y_true, (prob_pos >= t).astype(int), pos_label=1, zero_division=0)
        if f1 > best_f1:
            best_f1, best_t = f1, float(t)
    return best_t, best_f1


def make_folds(y, cfg, groups=None):
    if cfg.group_col and groups is not None:
        sgkf = StratifiedGroupKFold(n_splits=cfg.n_folds, shuffle=True, random_state=cfg.seed)
        return list(sgkf.split(np.zeros(len(y)), y, groups))
    skf = StratifiedKFold(n_splits=cfg.n_folds, shuffle=True, random_state=cfg.seed)
    return list(skf.split(np.zeros(len(y)), y))


def _softmax_pos(logits):
    z = logits - logits.max(axis=1, keepdims=True)
    e = np.exp(z)
    return (e / e.sum(axis=1, keepdims=True))[:, 1]

## 5.1. Tiện ích lưu / nạp **checkpoint** (trọng số tốt nhất)

Mỗi nhánh lưu mô hình của **fold có F1 validation cao nhất**: encoder dùng `save_model`, còn LR/GBDT/meta được pickle. Kèm `manifest.json` và `checkpoints_task*.zip`.


In [ ]:
import pickle, glob as _glob

def _ckpt_enabled(cfg):
    return bool(getattr(cfg, 'save_checkpoints', False))

def _ckpt_dir(cfg):
    d = getattr(cfg, 'checkpoint_dir', None) or os.path.join(cfg.output_dir, 'checkpoints')
    os.makedirs(d, exist_ok=True)
    return d

def save_pickle(obj, path):
    with open(path, 'wb') as f:
        pickle.dump(obj, f)
    return path

def load_pickle(path):
    with open(path, 'rb') as f:
        return pickle.load(f)

def write_json(d, path):
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(d, f, ensure_ascii=False, indent=2)
    return path

def fold_f1(y_true, prob_pos):
    return float(f1_score(y_true, (np.asarray(prob_pos) >= 0.5).astype(int),
                          pos_label=1, zero_division=0))


## 6. Branch B — encoder fine-tune (RoBERTa-BNE / XLM-R)
Xương sống. Cần GPU. Hỗ trợ chạy CV-only khi chưa có test.

In [18]:
def encoder_branch(train_texts, y, test_texts, folds, cfg,
                   text_pairs_train=None, text_pairs_test=None):
    import tempfile, torch
    from torch.utils.data import Dataset
    from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                              Trainer, TrainingArguments)
    tokenizer = AutoTokenizer.from_pretrained(cfg.encoder_model)

    class DS(Dataset):
        def __init__(self, a, b=None, labels=None):
            self.enc = tokenizer(a, b, truncation=True, max_length=cfg.max_len, padding="max_length")
            self.labels = labels
        def __len__(self): return len(self.enc["input_ids"])
        def __getitem__(self, i):
            item = {k: torch.tensor(v[i]) for k, v in self.enc.items()}
            if self.labels is not None: item["labels"] = torch.tensor(int(self.labels[i]))
            return item

    has_test = len(test_texts) > 0
    test_ds = DS(test_texts, text_pairs_test) if has_test else None
    seeds = getattr(cfg, "encoder_seeds", None) or [cfg.seed]
    oof = np.zeros(len(train_texts)); test_acc = np.zeros(len(test_texts) if has_test else 0)
    ckpt_on = _ckpt_enabled(cfg); ck = _ckpt_dir(cfg) if ckpt_on else None
    best_f1 = -1.0; best_info = None

    for sd in seeds:
        for fi, (tr, va) in enumerate(folds):
            print(f"[encoder] seed {sd} | fold {fi+1}/{len(folds)}")
            tr_a = [train_texts[i] for i in tr]; va_a = [train_texts[i] for i in va]
            tr_b = [text_pairs_train[i] for i in tr] if text_pairs_train else None
            va_b = [text_pairs_train[i] for i in va] if text_pairs_train else None
            tr_ds, va_ds = DS(tr_a, tr_b, y[tr]), DS(va_a, va_b)

            model = AutoModelForSequenceClassification.from_pretrained(cfg.encoder_model, num_labels=2)
            args = TrainingArguments(
                output_dir=os.path.join(tempfile.gettempdir(), f"enc_s{sd}_f{fi}"),  # /tmp -> khong rac outputs/
                num_train_epochs=cfg.epochs, per_device_train_batch_size=cfg.batch_size,
                per_device_eval_batch_size=cfg.batch_size*2, learning_rate=cfg.lr,
                warmup_ratio=0.06, weight_decay=0.01, logging_steps=50,
                save_strategy="no", report_to="none",
                fp16=torch.cuda.is_available(), seed=sd,
            )
            trainer = Trainer(model=model, args=args, train_dataset=tr_ds)
            trainer.train()
            va_pos = _softmax_pos(trainer.predict(va_ds).predictions)
            oof[va] += va_pos
            if has_test:
                test_acc += _softmax_pos(trainer.predict(test_ds).predictions)
            if ckpt_on:
                f1 = fold_f1(y[va], va_pos)
                if f1 > best_f1:
                    best_f1 = f1
                    bdir = os.path.join(ck, "encoder_best")
                    trainer.save_model(bdir); tokenizer.save_pretrained(bdir)
                    best_info = {"branch": "encoder", "best_seed": int(sd),
                                 "best_fold": int(fi + 1), "best_fold_f1": float(f1),
                                 "path": bdir, "model": cfg.encoder_model, "max_len": cfg.max_len}
                    print(f"[ckpt] encoder fold tot hon: F1={f1:.4f}")
            del model, trainer
            if torch.cuda.is_available(): torch.cuda.empty_cache()

    if ckpt_on and best_info:
        write_json(best_info, os.path.join(ck, "encoder_best.json"))
        print(f"[ckpt] encoder BEST fold F1={best_f1:.4f} -> {best_info['path']}")
    n_s = len(seeds)
    oof /= n_s
    return oof, (test_acc / (len(folds) * n_s) if has_test else np.zeros(0))


## 7. Branch C — perplexity + stylometry → GBDT (cho ST2)

In [19]:
_PUNCT = re.compile(r"[^\w\s]", re.UNICODE)

def stylometric_features(texts):
    rows = []
    for t in texts:
        t = str(t); w = t.split(); nw, nc = max(len(w),1), max(len(t),1)
        rows.append({
            "n_chars": len(t), "n_words": len(w),
            "avg_word_len": np.mean([len(x) for x in w]) if w else 0.0,
            "type_token_ratio": len(set(x.lower() for x in w))/nw,
            "punct_ratio": len(_PUNCT.findall(t))/nc,
            "upper_ratio": sum(c.isupper() for c in t)/nc,
            "digit_ratio": sum(c.isdigit() for c in t)/nc,
            "excl_count": t.count("!"), "ques_count": t.count("?"), "comma_count": t.count(","),
        })
    return pd.DataFrame(rows)

def perplexity_features(texts, model_id, max_len=256):
    import torch
    from transformers import AutoTokenizer, AutoModelForCausalLM
    tok = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForCausalLM.from_pretrained(model_id)
    device = "cuda" if torch.cuda.is_available() else "cpu"; model.to(device).eval()
    ppls, nll = [], []
    with torch.no_grad():
        for t in texts:
            ids = tok(str(t), return_tensors="pt", truncation=True, max_length=max_len)
            ids = {k: v.to(device) for k, v in ids.items()}
            if ids["input_ids"].shape[1] < 2: ppls.append(1e4); nll.append(10.0); continue
            loss = float(model(**ids, labels=ids["input_ids"]).loss)
            nll.append(loss); ppls.append(float(math.exp(min(loss, 20))))
    return pd.DataFrame({"perplexity": ppls, "mean_nll": nll})

def detection_branch(train_texts, y, test_texts, folds, cfg):
    print("[detector] perplexity + stylometric ...")
    Xtr = pd.concat([perplexity_features(train_texts, cfg.ppl_model),
                     stylometric_features(train_texts)], axis=1).to_numpy()
    has_test = len(test_texts) > 0
    Xte = (pd.concat([perplexity_features(test_texts, cfg.ppl_model),
                      stylometric_features(test_texts)], axis=1).to_numpy()
           if has_test else None)
    def new_model():
        # GBDT tu cai dat (Newton boosting) - KHONG dung LightGBM / sklearn
        return GradientBoostingClassifier(
            n_estimators=400, learning_rate=0.05, max_depth=4, min_samples_leaf=20,
            reg_lambda=1.0, subsample=0.8, colsample_bytree=0.8, random_state=cfg.seed)
    oof = np.zeros(len(train_texts)); test_acc = np.zeros(len(test_texts) if has_test else 0)
    ckpt_on = _ckpt_enabled(cfg); ck = _ckpt_dir(cfg) if ckpt_on else None
    best_f1 = -1.0; best_clf = None; best_fold = -1
    for fi, (tr, va) in enumerate(folds):
        clf = new_model(); clf.fit(Xtr[tr], y[tr])
        pv = clf.predict_proba(Xtr[va])[:, 1]; oof[va] = pv
        if has_test: test_acc += clf.predict_proba(Xte)[:, 1]
        if ckpt_on:
            f1 = fold_f1(y[va], pv)
            if f1 > best_f1: best_f1, best_clf, best_fold = f1, clf, fi + 1
    if ckpt_on and best_clf is not None:
        p = save_pickle(best_clf, os.path.join(ck, "detector_best.pkl"))
        write_json({"branch": "detector", "best_fold": int(best_fold),
                    "best_fold_f1": float(best_f1), "path": p,
                    "feature_dim": int(Xtr.shape[1]), "ppl_model": cfg.ppl_model},
                   os.path.join(ck, "detector_best.json"))
        print(f"[ckpt] detector BEST fold F1={best_f1:.4f} -> {p}")
    return oof, (test_acc/len(folds) if has_test else np.zeros(0))

## 7b. Branch D — TF-IDF + Logistic Regression (mạnh cho ST1)
Rẻ, không cần GPU. Stacking sẽ tự cân trọng số với encoder.

In [ ]:
from scipy.sparse import hstack   # chi muon cau truc ma tran thua (khong phai sklearn)
# TfidfVectorizer & LogisticRegression: dung ban tu cai dat o muc 5.0

def tfidf_branch(train_texts, y, test_texts, folds, cfg, C=4.0):
    has_test = len(test_texts) > 0
    word = TfidfVectorizer(analyzer="word", ngram_range=(1, 2), min_df=2,
                           sublinear_tf=True, strip_accents="unicode")
    char = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=2, sublinear_tf=True)
    Xtr = hstack([word.fit_transform(train_texts), char.fit_transform(train_texts)]).tocsr()
    Xte = hstack([word.transform(test_texts), char.transform(test_texts)]).tocsr() if has_test else None
    oof = np.zeros(len(train_texts)); ta = np.zeros(len(test_texts) if has_test else 0)
    ckpt_on = _ckpt_enabled(cfg); ck = _ckpt_dir(cfg) if ckpt_on else None
    best_f1 = -1.0; best_clf = None; best_fold = -1
    for fi, (tr, va) in enumerate(folds):
        clf = LogisticRegression(C=C, max_iter=2000, class_weight="balanced").fit(Xtr[tr], y[tr])
        pv = clf.predict_proba(Xtr[va])[:, 1]; oof[va] = pv
        if has_test: ta += clf.predict_proba(Xte)[:, 1]
        if ckpt_on:
            f1 = fold_f1(y[va], pv)
            if f1 > best_f1: best_f1, best_clf, best_fold = f1, clf, fi + 1
    if ckpt_on and best_clf is not None:
        bundle = {"lr": best_clf, "word_vectorizer": word, "char_vectorizer": char, "C": C}
        p = save_pickle(bundle, os.path.join(ck, "tfidf_best.pkl"))
        write_json({"branch": "tfidf", "best_fold": int(best_fold),
                    "best_fold_f1": float(best_f1), "path": p,
                    "vocab_word": len(word.vocabulary_), "vocab_char": len(char.vocabulary_)},
                   os.path.join(ck, "tfidf_best.json"))
        print(f"[ckpt] tfidf BEST fold F1={best_f1:.4f} -> {p}")
    return oof, (ta / len(folds) if has_test else np.zeros(0))


## 8. Branch A — kNN-retrieval few-shot + self-consistency (tùy chọn)
Trỏ `llm_model` tới checkpoint Gemma của bạn rồi đặt `use_llm_branch=True`.

In [20]:
def _retrieve_few_shot(qvec, pool_vecs, pool_y, k_per_class, exclude_idx=None):
    sims = pool_vecs @ qvec
    if exclude_idx is not None: sims[exclude_idx] = -1e9
    chosen = []
    for cls in (1, 0):
        ci = np.where(pool_y == cls)[0]
        chosen.extend(ci[np.argsort(-sims[ci])][:k_per_class].tolist())
    return chosen

def _build_prompt(qtext, shots, pool_texts, pool_y, cfg):
    pos, neg = cfg.out_positive, cfg.out_negative
    lines = [f"Eres un clasificador experto. Clasifica cada texto como '{pos}' o '{neg}'.",
             "Razona brevemente y termina con una linea EXACTA: Respuesta: <clase>.", "", "Ejemplos:"]
    for i in shots:
        lines.append(f"Texto: {pool_texts[i]}\nRespuesta: {pos if pool_y[i]==1 else neg}\n")
    lines.append(f"Ahora clasifica:\nTexto: {qtext}\nRazonamiento:")
    return "\n".join(lines)

def llm_branch(train_texts, y, test_texts, cfg):
    from sentence_transformers import SentenceTransformer
    import torch
    from transformers import AutoTokenizer, AutoModelForCausalLM
    retr = SentenceTransformer(cfg.retriever_model)
    pool_vecs = retr.encode(train_texts, normalize_embeddings=True, show_progress_bar=True)
    test_vecs = retr.encode(test_texts, normalize_embeddings=True, show_progress_bar=True) if test_texts else []
    tok = AutoTokenizer.from_pretrained(cfg.llm_model)
    llm = AutoModelForCausalLM.from_pretrained(cfg.llm_model, torch_dtype="auto", device_map="auto")

    def generate(prompt, n):
        ids = tok.apply_chat_template([{"role": "user", "content": prompt}],
                                      add_generation_prompt=True, return_tensors="pt").to(llm.device)
        outs = []
        for _ in range(n):
            gen = llm.generate(ids, max_new_tokens=120, do_sample=True, temperature=0.35, top_p=0.95)
            outs.append(tok.decode(gen[0][ids.shape[1]:], skip_special_tokens=True))
        return outs

    def parse(text):
        m = re.search(r"respuesta\s*:\s*([a-zaeioun]+)", text.lower())
        if not m: return None
        ans = m.group(1)
        if cfg.out_positive.lower().startswith(ans[:4]): return 1
        if cfg.out_negative.lower().startswith(ans[:4]): return 0
        return None

    def predict_one(qtext, qvec, exclude=None):
        shots = _retrieve_few_shot(qvec, pool_vecs, y, cfg.few_shot_k_per_class, exclude)
        votes = [parse(t) for t in generate(_build_prompt(qtext, shots, train_texts, y, cfg), cfg.self_consistency_n)]
        votes = [v for v in votes if v is not None]
        return float(np.mean(votes)) if votes else 0.5

    print("[llm] scoring train (OOF) ...")
    oof = np.array([predict_one(train_texts[i], pool_vecs[i], i) for i in range(len(train_texts))])
    test = np.array([predict_one(test_texts[i], test_vecs[i]) for i in range(len(test_texts))]) if test_texts else np.zeros(0)
    return oof, test

## 9. Stacking + tinh chỉnh ngưỡng + xuất file nộp

In [21]:
def stack_and_threshold(branch_oof, branch_test, y, have_test=True, cfg=None):
    names = sorted(branch_oof)
    if not names:
        raise RuntimeError("Khong co nhanh nao chay.")
    X_oof = np.column_stack([branch_oof[n] for n in names])
    if len(names) == 1:
        oof_p = X_oof[:, 0]
        test_p = branch_test[names[0]] if have_test else np.zeros(0)
    else:
        meta = LogisticRegression(max_iter=1000, class_weight="balanced")
        meta.fit(X_oof, y)
        oof_p = meta.predict_proba(X_oof)[:, 1]
        print("[stack] trong so meta:", dict(zip(names, np.round(meta.coef_[0], 3))))
        if have_test:
            X_test = np.column_stack([branch_test[n] for n in names])
            test_p = meta.predict_proba(X_test)[:, 1]
        else:
            test_p = np.zeros(0)
    thr, best_f1 = tune_threshold(y, oof_p)
    f1_05 = f1_score(y, (oof_p >= 0.5).astype(int), pos_label=1, zero_division=0)
    print(f"[stack] OOF F1@0.5 = {f1_05:.4f} | OOF F1(thr={thr:.3f}) = {best_f1:.4f}")
    pred = (test_p >= thr).astype(int) if have_test else np.zeros(0, dtype=int)
    if cfg is not None and _ckpt_enabled(cfg):
        ck = _ckpt_dir(cfg)
        meta_obj = meta if len(names) > 1 else None
        save_pickle({"meta": meta_obj, "branch_names": names,
                     "threshold": float(thr), "single_branch": len(names) == 1},
                    os.path.join(ck, "stacking_best.pkl"))
        write_json({"branches": names, "threshold": float(thr),
                    "oof_f1": float(best_f1), "oof_f1_at_0.5": float(f1_05),
                    "meta_coef": (dict(zip(names, [float(x) for x in meta.coef_[0]]))
                                  if len(names) > 1 else None)},
                   os.path.join(ck, "stacking_best.json"))
        print(f"[ckpt] stacking meta + nguong -> {ck}")
    return pred, thr, best_f1


def write_submission(test_df, pred, cfg):
    os.makedirs(cfg.output_dir, exist_ok=True)
    ids = test_df[cfg.id_col] if cfg.id_col in test_df.columns else pd.RangeIndex(len(test_df))
    labels = np.where(pred == 1, cfg.out_positive, cfg.out_negative)
    sub = pd.DataFrame({cfg.id_col: ids, "tag": labels})

    # ten file dung chuan: task1.tsv / task2.tsv
    tsv_name = f"task{cfg.subtask}.tsv"
    tsv_path = os.path.join(cfg.output_dir, tsv_name)
    # TSV: tab-separated. Du lieu khong co tab/newline nen khong can quoting
    sub.to_csv(tsv_path, sep="\t", index=False)

    # kiem tra: du dong + dung thu tu id
    assert len(sub) == len(test_df), "Thieu/ thua dong so voi test set!"
    assert list(sub[cfg.id_col]) == list(test_df[cfg.id_col]), "Sai thu tu id!"

    zip_path = os.path.join(cfg.output_dir, f"submission_task{cfg.subtask}.zip")
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        zf.write(tsv_path, arcname=tsv_name)   # zip phai chua dung 'task1.tsv'
    print(f"[io] {tsv_path}  ({len(sub)} dong) | phan bo: {sub['tag'].value_counts().to_dict()}")
    print(f"[io] {zip_path}  (chua '{tsv_name}')")

## 10. Chạy toàn bộ
Nếu chưa có file test, ô này chỉ chạy CV và in F1 (bỏ qua nộp). Kết quả ở `/kaggle/working/outputs/`.

In [22]:
def run(cfg):
    set_seed(cfg.seed); os.makedirs(cfg.output_dir, exist_ok=True)
    train_df = pd.concat([load_table(p) for p in cfg.train_paths], ignore_index=True)
    have_test = bool(cfg.test_path) and os.path.exists(cfg.test_path)
    test_df = load_table(cfg.test_path) if have_test else None
    print(f"[data] train = {len(train_df)} | test = {len(test_df) if have_test else 'KHONG CO -> CV-only'}")

    y = encode_labels(train_df, cfg)
    print(f"[data] positive '{cfg.positive_label}' = {int(y.sum())}/{len(y)}")
    groups = train_df[cfg.group_col].to_numpy() if cfg.group_col else None
    folds = make_folds(y, cfg, groups)
    print(f"[data] split = {'StratifiedGroupKFold('+cfg.group_col+')' if cfg.group_col else 'StratifiedKFold'} x{cfg.n_folds}")

    if len(cfg.text_cols) == 2:
        train_texts = train_df[cfg.text_cols[0]].astype(str).tolist()
        pair_tr     = train_df[cfg.text_cols[1]].astype(str).tolist()
        test_texts  = test_df[cfg.text_cols[0]].astype(str).tolist() if have_test else []
        pair_te     = test_df[cfg.text_cols[1]].astype(str).tolist() if have_test else None
    else:
        train_texts = build_text(train_df, cfg)
        test_texts  = build_text(test_df, cfg) if have_test else []
        pair_tr = pair_te = None

    boof, btest = {}, {}
    if cfg.use_encoder_branch:
        print("\n=== Branch B: encoder ===")
        boof["encoder"], btest["encoder"] = encoder_branch(train_texts, y, test_texts, folds, cfg, pair_tr, pair_te)
    if cfg.use_detection_branch:
        print("\n=== Branch C: detector ===")
        det_tr = train_df[cfg.text_cols[-1]].astype(str).tolist()
        det_te = test_df[cfg.text_cols[-1]].astype(str).tolist() if have_test else []
        boof["detector"], btest["detector"] = detection_branch(det_tr, y, det_te, folds, cfg)
    if getattr(cfg, "use_tfidf_branch", False):
        print("\n=== Branch D: TF-IDF + LR ===")
        tf_tr = build_text(train_df, cfg)
        tf_te = build_text(test_df, cfg) if have_test else []
        boof["tfidf"], btest["tfidf"] = tfidf_branch(tf_tr, y, tf_te, folds, cfg)
    if cfg.use_llm_branch:
        print("\n=== Branch A: LLM ===")
        boof["llm"], btest["llm"] = llm_branch(build_text(train_df, cfg), y,
                                               build_text(test_df, cfg) if have_test else [], cfg)

    print("\n=== Stacking ===")
    pred, thr, f1 = stack_and_threshold(boof, btest, y, have_test=have_test, cfg=cfg)
    if have_test:
        write_submission(test_df, pred, cfg)
        print(f"[check] ty le du doan '{cfg.out_positive}' tren test = {pred.mean():.2f} (train ~0.50)")
    else:
        print("[io] Chua co test -> bo qua nop. OOF F1 o tren la uoc luong tin cay.")
    if _ckpt_enabled(cfg):
        ck = _ckpt_dir(cfg)
        per = {}
        for jp in _glob.glob(os.path.join(ck, "*_best.json")):
            try: per[os.path.basename(jp)] = json.load(open(jp, encoding="utf-8"))
            except Exception: pass
        manifest = {"subtask": cfg.subtask, "positive_label": cfg.positive_label,
                    "branches_used": sorted(boof.keys()),
                    "oof_f1": float(f1), "threshold": float(thr),
                    "encoder_model": cfg.encoder_model, "details": per}
        write_json(manifest, os.path.join(ck, "manifest.json"))
        zip_path = os.path.join(cfg.output_dir, f"checkpoints_task{cfg.subtask}.zip")
        with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
            for root, _, files in os.walk(ck):
                if os.path.relpath(root, ck).startswith("encoder_best"): continue
                for fn in files:
                    fp = os.path.join(root, fn)
                    zf.write(fp, arcname=os.path.relpath(fp, ck))
        print(f"[ckpt] checkpoint da luu o: {ck}")
        enc = os.path.join(ck, "encoder_best")
        if os.path.isdir(enc): print(f"[ckpt] trong so encoder tot nhat (nang): {enc}")
        print(f"[ckpt] zip cac model nhe: {zip_path}")
    print(f"\nXong. OOF F1 (lop '{cfg.positive_label}') = {f1:.4f}, nguong = {thr:.3f}")

run(CFG)

[data] train = 362 | test = 550
[data] positive 'machine' = 181/362
[data] split = StratifiedGroupKFold(headline) x5

=== Branch B: encoder ===
[encoder] fold 1/5


Some weights of XLMRobertaForSequenceClassification were not initialized from the model checkpoint at xlm-roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Step,Training Loss


Some weights of XLMRobertaForSequenceClassification were not initialized from the model checkpoint at xlm-roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


[encoder] fold 2/5


...

special_tokens_map.json:   0%|          | 0.00/262 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/261M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/261M [00:00<?, ?B/s]

`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMCl


=== Stacking ===
[stack] trong so meta: {'detector': np.float64(3.825), 'encoder': np.float64(0.47)}
[stack] OOF F1@0.5 = 0.8365 | OOF F1(thr=0.650) = 0.8531
[io] da ghi /kaggle/working/outputs/submission.csv  (550 dong) | phan bo: {'human': 288, 'machine': 262}
[io] da ghi /kaggle/working/outputs/submission.zip
[check] ty le du doan 'machine' tren test = 0.48 (train ~0.50)

Xong. OOF F1 (lop 'machine') = 0.8531, nguong = 0.650


## 11. Nộp & lưu ý
- File nộp ở `/kaggle/working/outputs/` (`submission.csv` + `submission.zip`). Tải về từ tab **Output** rồi nộp lên Codabench.
- **Định dạng:** mình để `id,tag` dạng CSV. File gold gốc là TSV `id\ttag` — hãy đối chiếu với starting kit; nếu cần TSV thì đổi `sub.to_csv(..., sep="\t")`.
- **Phân bố:** train cân bằng 50/50, test chưa chắc. Nếu `ty le du doan` lệch nhiều so với 0.50 mà bạn tin test cân bằng, cân nhắc dùng ngưỡng gần 0.5 (so sánh `F1@0.5` vs `F1(thr)`).

### Ăn điểm thêm
- **Task 2:** giữ GroupKFold; tín hiệu nên đến từ *joke* (perplexity/stylometric) vì test toàn headline lạ.
- Đổi `encoder_model` sang `xlm-roberta-large` / `microsoft/mdeberta-v3-base`, mỗi cái là một nhánh để stacking tự cân.
- Nâng Branch C lên **Binoculars** (tỉ lệ perplexity 2 mô hình).
- Chạy encoder với vài `seed` rồi trung bình (BERT variance cao trên dữ liệu nhỏ).

---

## 12. Checkpoint trọng số tốt nhất

Khi `cfg.save_checkpoints=True` (mặc định), sau `run(CFG)` notebook lưu vào `<output_dir>/checkpoints/`:

- `encoder_best/` — trọng số encoder của **fold có F1 cao nhất** (`save_model` + tokenizer); nạp lại bằng `AutoModelForSequenceClassification.from_pretrained(path)`.
- `detector_best.pkl` — GBDT tốt nhất; `tfidf_best.pkl` — `{lr, word_vectorizer, char_vectorizer}` tốt nhất; `stacking_best.pkl` — `{meta, branch_names, threshold}`.
- `*_best.json` ghi fold + F1 của từng nhánh; `manifest.json` tóm tắt; `checkpoints_task*.zip` gói các model nhẹ (không gồm encoder vì nặng).

Nạp lại nhanh một nhánh nhẹ:

```python
b = load_pickle('/kaggle/working/outputs/checkpoints/tfidf_best.pkl')
from scipy.sparse import hstack
X = hstack([b['word_vectorizer'].transform(texts),
            b['char_vectorizer'].transform(texts)]).tocsr()
prob = b['lr'].predict_proba(X)[:, 1]
st = load_pickle('/kaggle/working/outputs/checkpoints/stacking_best.pkl')
pred = (prob >= st['threshold']).astype(int)
```

Tắt lưu checkpoint: đặt `CFG.save_checkpoints = False`. Đổi nơi lưu: `CFG.checkpoint_dir = '...'`.
